# Threads & concurrence · **côté Python**

*Piste Python / Colab — analogue de `3_threads_et_concurrence.jl`.*

On **mesure** ce que le **GIL** fait (et ne fait pas), on déclenche une **race condition**, puis on
parallélise vraiment avec des **processus**.

> Sur Colab : start method `fork` (Linux) → les `ProcessPool` marchent ; les fonctions doivent être
> définies au niveau module (elles le sont ici).

In [ ]:
import timeit

def best_time(fn, repeat=7):
    """Temps par appel (s), façon %timeit : auto-échelle puis min sur `repeat` essais."""
    number = 1
    while timeit.timeit(fn, number=number) < 0.05:
        number *= 10
    return min(timeit.repeat(fn, number=number, repeat=repeat)) / number

def ms(t): return f"{t*1e3:.3f} ms"
def us(t): return f"{t*1e6:.2f} µs"


In [ ]:
import numpy as np, time
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def travail(x):
    s = 0.0
    for k in range(1, 201):
        s += np.sin(x + k) * np.cos(x - k)
    return s

# fonction CPU-bound pure Python (volontairement non vectorisée)
def charge(rep):
    s = 0.0
    for i in range(rep):
        s += (i % 7) * 1.0000001
    return s
ITER = 5_000_000

## 1. Le GIL : `threading` n'accélère pas le CPU

CPython n'exécute **qu'un seul thread de bytecode à la fois** (Global Interpreter Lock). Sur du calcul
CPU pur, les threads se relaient sans jamais tourner en parallèle. Les **processus**, eux, ont chacun
leur interpréteur → vrai parallélisme (au prix de la copie de données).

In [ ]:
def run(executor_cls, workers, tasks):
    with executor_cls(max_workers=workers) as ex:
        list(ex.map(charge, tasks))

tasks = [ITER]*8

t_seq = best_time(lambda: [charge(t) for t in tasks], repeat=3)
t_thr = best_time(lambda: run(ThreadPoolExecutor, 8, tasks), repeat=3)
t_prc = best_time(lambda: run(ProcessPoolExecutor, 8, tasks), repeat=3)

print("séquentiel        :", ms(t_seq))
print("8 threads (GIL)   :", ms(t_thr), f"→ accélération {t_seq/t_thr:.2f}×  (≈ 1 : aucun gain)")
print("8 processus       :", ms(t_prc), f"→ accélération {t_seq/t_prc:.2f}×  (vrai parallélisme)")

**Verdict.** Les threads ≈ le séquentiel (le GIL sérialise) ; les processus accélèrent ~×cœurs.

> Julia n'a **pas** de GIL : `Threads.@threads` parallélise vraiment en mémoire partagée. Plus
> puissant… et c'est ce qui ouvre la porte aux *race conditions* ci-dessous.

## 2. La race condition

`compteur += 1` n'est **pas** atomique : c'est lire → ajouter → réécrire, sur plusieurs bytecodes.

Nuance honnête sur le GIL : sur CPython, une boucle serrée **finit souvent avant toute préemption**,
ce qui *masque* le bug — mais ne le **corrige pas**. Pour rendre l'entrelacement visible, on insère
`time.sleep(0)` entre la lecture et l'écriture : il **rend la main** à un autre thread au pire moment.
C'est une exécution que l'ordonnanceur a **toujours le droit** de produire — et que la version
**sans-GIL** de Python (≥ 3.13) produit *sans aucun* `sleep`.

In [ ]:
import threading, time

compteur = 0
def incr(n):
    global compteur
    for _ in range(n):
        v = compteur
        time.sleep(0)          # rend la main : expose l'entrelacement (toujours permis)
        compteur = v + 1       # réécrit une valeur périmée → des incréments sont perdus

K, T = 2000, 8
compteur = 0
threads = [threading.Thread(target=incr, args=(K,)) for _ in range(T)]
for t in threads: t.start()
for t in threads: t.join()

print("attendu :", K*T)
print("obtenu  :", compteur, " → des incréments perdus, et le nombre varie d'une exécution à l'autre")

## 3. Réparer : verrou, ou accumulateurs séparés

Comme en Julia, la bonne approche est de **ne pas partager** dans la boucle chaude : chaque worker
accumule de son côté, on combine à la fin (ici via des processus).

In [ ]:
# correct : un verrou rend l'incrément indivisible (mais sérialise → lent)
compteur = 0
lock = threading.Lock()
def incr_lock(n):
    global compteur
    for _ in range(n):
        with lock:
            compteur += 1

compteur = 0
threads = [threading.Thread(target=incr_lock, args=(K,)) for _ in range(T)]
for t in threads: t.start()
for t in threads: t.join()
print("avec verrou — attendu :", K*T, " obtenu :", compteur)

## 4. Exemple conducteur : Monte-Carlo de π en parallèle (processus)

In [ ]:
def pi_chunk(n):
    rng = np.random.default_rng()
    x = rng.random(n); y = rng.random(n)
    return int(np.count_nonzero(x*x + y*y <= 1.0))

def pi_parallel(n, workers=8):
    part = n // workers
    with ProcessPoolExecutor(max_workers=workers) as ex:
        c = sum(ex.map(pi_chunk, [part]*workers))
    return 4*c/(part*workers)

def pi_seq(n):
    return 4*pi_chunk(n)/n

n = 50_000_000
print("π (réf) :", np.pi)
print("séquentiel :", ms(best_time(lambda: pi_seq(n), repeat=3)))
print("processus  :", ms(best_time(lambda: pi_parallel(n), repeat=3)))

## Bilan — à comparer avec Julia

| | Python | Julia |
|---|---|---|
| threads sur CPU pur | **aucun gain** (GIL) | vrai parallélisme (`@threads`) |
| paralléliser | **processus** (copie de données) | threads (mémoire partagée) |
| race condition | possible (`+=` non atomique) | possible (et silencieuse) |

> Même leçon des deux côtés : **minimiser le partage**. Python l'impose presque (processus isolés) ;
> Julia vous laisse partager la mémoire — puissant, mais à vos risques.